<a href="https://colab.research.google.com/github/adidror005/Sefaria/blob/main/Rambam_RAG_with_LLama_Index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import base64
from IPython.display import Image, display

def mm_ink(graph: str):
    graphbytes = graph.encode("utf-8")
    base64_string = base64.urlsafe_b64encode(graphbytes).decode("ascii")
    return "https://mermaid.ink/img/" + base64_string

def mm(graph: str):
    display(Image(url=mm_ink(graph)))

In [4]:
import pandas as pd
df = pd.read_parquet("rambam_df.parquet")
df.groupby('hilchot').head(1)

,ref,book,hilchot,chapter,halacha,mitzvah_num,intro_num,section_type,text
0,"Mishneh Torah, Transmission of the Oral Law 1",Mishneh Torah,Transmission of the Oral Law,NaN,NaN,NaN,1.0,intro,"<b>The Rambam's Introduction</b><sup class=""fo..."
45,"Mishneh Torah, Positive Mitzvot 1",Mishneh Torah,Positive Mitzvot,NaN,NaN,1.0,NaN,positive_mitzvah,The first of the positive commandments is the ...
293,"Mishneh Torah, Negative Mitzvot 1",Mishneh Torah,Negative Mitzvot,NaN,NaN,1.0,NaN,negative_mitzvah,The first mitzvah of the negative commandments...
663,"Mishneh Torah, Overview of Mishneh Torah Conte...",Mishneh Torah,Overview of Mishneh Torah Contents,NaN,NaN,NaN,1.0,overview,I have seen fit to divide this work into fourt...
677,"Mishneh Torah, Foundations of the Torah 1:1",Mishneh Torah,Foundations of the Torah,1.0,1.0,NaN,NaN,halacha,The foundation of all foundations and the pill...
...,...,...,...,...,...,...,...,...,...
14931,"Mishneh Torah, The Sanhedrin and the Penalties...",Mishneh Torah,The Sanhedrin and the Penalties within Their J...,1.0,1.0,NaN,NaN,halacha,It is a positive Scriptural commandment to app...
15185,"Mishneh Torah, Testimony 1:1",Mishneh Torah,Testimony,1.0,1.0,NaN,NaN,halacha,A witness is commanded to testify in court wit...
15358,"Mishneh Torah, Rebels 1:1",Mishneh Torah,Rebels,1.0,1.0,NaN,NaN,halacha,The Supreme <i>Sanhedrin</i> in Jerusalem are ...
15427,"Mishneh Torah, Mourning 1:1",Mishneh Torah,Mourning,1.0,1.0,NaN,NaN,halacha,It is a positive commandment to mourn for one'...


| ref | sefer | hilchot | chapter | halacha | section_type | text |
|---|---|---|---:|---:|---|---|
| Mishneh Torah, Teshuva 2:1 | Sefer Mada | Teshuva | 2 | 1 | halacha | What is complete repentance?... |
| Mishneh Torah, Transmission of the Oral Law 1 | Introduction | Transmission of the Oral Law | NaN | NaN | intro | Moses received the Torah... |

---

## 📖 Why Mishneh Torah Works Well for RAG

*Mishneh Torah* is highly structured:

1. **Section)**  
2. **Chapter**  
3. **Halacha (Law)**  

Each halacha is:
- Short  
- Self-contained  
- Semantically clear  

This makes it a natural **retrieval unit**.

At the same time, the hierarchy provides rich metadata:
- `sefer`
- `hilchot`
- `chapter`
- `halacha`
- `section_type`

This enables:
- Filtering (e.g., only certain domains)
- Better ranking
- Clear source attribution

---

## 🎯 Key Idea

> Each row in the DataFrame becomes a retrieval unit, and the metadata preserves the full structure of Mishneh Torah — including introductions — allowing flexible and precise RAG behavior.



# What We’re Building

We want a chatbot that can answer questions like:

- What does Rambam say about repentance?
- What are the laws of charity?
- How does Rambam describe free will?

The difference between this and ChatGPT is we explicitly find the most relevant sources in the text and then our LLM must use those sources to give its answer.


# Big Picture: What Is RAG?

RAG stands for **Retrieval-Augmented Generation**.

Instead of asking the LLM to answer from memory:

1. Retrieve relevant sources based on semantic similarity (embedding similarity with query) or exact match filtering
2. Give those sources to the LLM
3. Generate an answer grounded in those sources

The key idea is:

> Retrieve first, generate second answer second.


In [12]:
# @title
from IPython.display import Markdown,display
display(Markdown("# RAG Flow for Rambam Q&A"))

mm("""
flowchart TD
    A["User Question<br/>What does Rambam say about repentance?"]
    A --> B["Embed Query<br/>q = [0.82, -0.14, 0.33, 0.71]"]
    B --> C["Vector Search<br/>Find K Most Similar Chunks"]

    D1["Rambam Chunk 1<br/>Hilchot Teshuva 2:1"]
    D2["Rambam Chunk 2<br/>Hilchot Deot 1:3"]
    D3["Rambam Chunk 3<br/>Hilchot Teshuva 7:4"]

    D1 --> C
    D2 --> C
    D3 --> C

    C --> E["Return Top K Matches"]
    E --> F["Prompt to LLM<br/>Question + Sources"]
    F --> G["Final Answer<br/>with citations"]
""")

# RAG Flow for Rambam Q&A

# Build a Rambam Q&A Chatbot with LlamaIndex + OpenAI + Gradio

In this notebook, we build a source-grounded Q&A chatbot over Rambam’s *Mishneh Torah*.

A separate video/notebook covers how to collect and structure the data from the Sefaria API. Here, we assume that step is already done and that we have a DataFrame `df`.

We will use:

- **LlamaIndex** for indexing and retrieval  
- **OpenAI** for embeddings and answer generation  
- **Gradio** for the chat interface  

---

## 📊 Assumed Input DataFrame

We assume `df` contains **one row per atomic unit** from *Mishneh Torah*.

In most cases, this is a **single halacha**, but introductory and special sections are also included and handled via metadata.

---

### Expected Columns

- `ref` — full textual reference  
- `sefer` — book of Mishneh Torah  
- `section name` — section within the sefer  
- `chapter` — chapter number (NaN for intros)  
- `halacha` — halacha number (NaN for intros)  
- `mitzvah_num` — for mitzvah listings (optional)  
- `intro_num` — for introduction segments (optional)  
- `section_type` — `"halacha"` or `"intro"`  
- `text` — text content  

---

### 🧠 Notes on Structure

- **Standard halachot:**
  - Have `chapter` and `halacha`
  - Represent the **main retrieval units**

- **Intro / special sections:**
  - Do not follow the standard hierarchy
  - Use `intro_num` or `mitzvah_num`
  - Have `section_type = "intro"`
  - `chapter` and `halacha` are `NaN`

👉 In RAG, these are **not treated differently structurally** —  
they are handled via **metadata filtering and routing**.

---

### 📌 Load DataFrame


# Installs Needed